# Tavily 수집 데이터 확인
각 검색 쿼리별로 어떤 내용이 수집되는지 확인합니다.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from tavily import TavilyClient
from datetime import datetime

client = TavilyClient(api_key=os.getenv('TAVILY_API_KEY'))

CATEGORY = '기초화장품'
COUNTRY = 'United States'
this_year = datetime.now().year
last_year = this_year - 1

print(f'카테고리: {CATEGORY} / 국가: {COUNTRY} / 기준연도: {last_year}~{this_year}')

카테고리: 기초화장품 / 국가: United States / 기준연도: 2025~2026


In [ ]:
# 쿼리 목록 - 미국/일본 화장품 온/오프라인 판매 비율 특화
queries = [
    f"United States beauty cosmetics online vs offline sales ratio percentage {last_year} {this_year} e-commerce share statistics",
    f"Japan beauty cosmetics online offline sales ratio EC market share percentage {last_year} {this_year} statistics",
    f"US skincare e-commerce penetration rate beauty industry online shopping share {last_year} {this_year}",
    f"Japan skincare EC ratio online offline beauty retail market share {last_year} {this_year}",
    f"K-beauty {CATEGORY} online offline channel sales share {COUNTRY} {last_year} {this_year} percentage statistics",
    f"K-beauty {CATEGORY} export statistics South Korea to {COUNTRY} {last_year}",
]

for i, q in enumerate(queries, 1):
    print(f'[{i}] {q}')

In [5]:
# 쿼리 인덱스 선택 (0~5)
QUERY_IDX = 0

query = queries[QUERY_IDX]
print(f'검색 쿼리: {query}\n')

results = client.search(query=query, search_depth='advanced', max_results=5)

for i, r in enumerate(results.get('results', []), 1):
    print(f'--- [{i}] {r["url"]} ---')
    print(r.get('content', '')[:500])
    print()

검색 쿼리: K-beauty 기초화장품 market size CAGR growth rate United States 2025 2026 report

--- [1] https://market.us/report/k-beauty-products-market/ ---
| Report Features | Description |
   --- |
  | Market Value (2025) | USD 110.2 Billion |
  | Forecast Revenue (2035) | USD 275.6 Billion |
  | CAGR (2026-2035) | 9.6% |
  | Base Year for Estimation | 2025 |
  | Historic Period | 2020-2024 |
  | Forecast Period | 2026-2035 |
  | Report Coverage | Revenue Forecast, Market Dynamics, Competitive Landscape, Recent Developments |
  | Segments Covered | By Product (Skin Care, Hair Care, Makeup, Others), By End-User (Women, Men), By Distribution Channel

--- [2] https://bio2laserstudio.com/the-rise-of-k-beauty-a-comprehensive-analysis-of-its-impact-on-the-us-market-in-2025/ ---
Korea’s total cosmetic exports in 2024(#ref-7). This aligns perfectly with a broader movement in the U.S. beauty industry, where the skincare market is projected to reach $31.5 billion by 2028, growing at a compound annual gro

In [ ]:
# 전체 쿼리 한번에 실행 - 소스 URL 목록만 확인
all_sources = {}

for i, query in enumerate(queries, 1):
    results = client.search(query=query, search_depth='advanced', max_results=5)
    urls = [r['url'] for r in results.get('results', [])]
    all_sources[i] = urls
    print(f'[쿼리 {i}] {query[:60]}...')
    for url in urls:
        print(f'  - {url}')
    print()

In [ ]:
# 특정 쿼리의 전체 content 확인
QUERY_IDX = 1  # 0~5 중 선택

results = client.search(query=queries[QUERY_IDX], search_depth='advanced', max_results=5)
for r in results.get('results', []):
    print(f'=== {r["url"]} ===')
    print(r.get('content', ''))
    print()

# AI 최적 국가 추천 — 점수 설계 문서

---

## 전체 AI Score 구성

```
AI Score (0~100) = 트렌드 적합도(25%) + 시장 규모 점수(50%) + 리뷰 유사도(25%)
```

---

## 1. 시장 규모 점수 (50%)

### 개요
국가별 시장 매력도를 5개 지표의 가중 합산으로 산출 (0~1 범위)

### 구성 지표 및 가중치

| 지표 | 가중치 | 데이터 출처 | 의미 |
|------|--------|-------------|------|
| 시장 규모 | 0.10 | `MarketResearch.market_size.value` | 절대적 기회 크기 |
| 시장 성장률(CAGR) | 0.25 | `MarketResearch.market_size.cagr` | 진입 타이밍 |
| K-뷰티 점유율 | 0.25 | `MarketResearch.kbeauty_share.share` | 시장 수용성 |
| 한국 화장품 수출 성장률 | 0.20 | `MarketStat.amount` (연도별) | 실제 판매 증거 |
| K-뷰티 광고 활동량 | 0.20 | `MetaAdSummary.total_ads` (채널 합산) | 브랜드 투자 활동 증거 |

### 계산 방법

**1. 각 지표 파싱**
- 시장 규모: `"$18.4B"` → `18.4` (단위: 십억 달러)
- 시장 성장률: `"6.96%"` → `0.0696`
- K-뷰티 점유율: `"15.8%"` → `0.158`
- 수출 성장률: 최근 2개년 YoY 성장률 평균
  ```
  YoY_1 = (2023년 수출액 - 2022년 수출액) / 2022년 수출액
  YoY_2 = (2024년 수출액 - 2023년 수출액) / 2023년 수출액
  수출 성장률 = (YoY_1 + YoY_2) / 2
  ```
- 광고 활동량: 국가별 채널(US: ulta+sephora, JP: qoo10+rakuten) `total_ads` 합산

**2. 국가 간 정규화 (상대 비율)**
```
국가A 정규화 값 = 국가A 원값 / (국가A 원값 + 국가B 원값)
```
→ 두 국가의 정규화 값 합계 = 1.0

**3. 가중 합산**
```
시장 점수 = 시장규모(0.10) + 성장률(0.25) + K뷰티점유율(0.25) + 수출성장률(0.20) + 광고활동(0.20)
```

### 예외 처리

| 상황 | 처리 방법 |
|------|----------|
| 데이터 파싱 실패 | 해당 지표 0.5 적용 (중립값) |
| 수출 성장률 음수 | 그대로 반영 (불리한 시장으로 처리) |
| 연도 데이터 1개만 있음 | YoY 1개로 대체 |
| 연도 데이터 없음 | 0.5 적용 (중립값) |
| MetaAdSummary 데이터 없음 | 0.5 적용 (중립값) |

---

## 2. 트렌드 적합도 (25%)

### 개요
사용자 제품의 성분·효능이 해당 국가의 시장 트렌드와 얼마나 부합하는지 GPT로 채점 (0~10 → 정규화)

### 사용 데이터

| 구분 | 데이터 | 출처 |
|------|--------|------|
| 사용자 측 | 주요 성분, 핵심 효능 | 사용자 입력 |
| 국가 측 | trends.ingredients, trends.functions, trends.details | `MarketResearch.trends` |

### 계산 방법

**1. GPT 채점 (temperature=0)**
```
입력: 사용자 성분 + 효능  vs  국가별 트렌드 (ingredients + functions + details)
출력 JSON:
{
  "trend_score": 7.5,          // 0~10
  "matched_keywords": ["CICA", "보습"],  // 매칭된 트렌드 키워드
  "reasoning": "병풀 추출물은 CICA와 동일..."  // 근거 텍스트
}
```

**2. 출력 검증**
```python
# GPT 반환 키워드가 실제 트렌드 목록에 있는지 교차 확인 (환각 방지)
verified_keywords = [k for k in gpt_result["matched_keywords"] if k in actual_trends]
```

**3. 정규화 (국가 간 상대 비율)**
```
국가A 정규화 값 = 국가A 점수 / (국가A 점수 + 국가B 점수)
```

### 예외 처리

| 상황 | 처리 방법 |
|------|----------|
| 두 나라 모두 트렌드 미매칭 | 낮은 점수 그대로 반영 (추후 개선 예정) |
| GPT 응답 파싱 실패 | 0.5 중립값 적용 |
| trends 데이터 없음 | 0.5 중립값 적용 |

### 부산물
- `reasoning` 텍스트 → 최종 선정 근거 생성 시 재활용

---

## 3. 리뷰 유사도 (25%)

### 개요
국가별 Top10 상품 리뷰 데이터를 기반으로, 해당 국가 소비자의 관심사와 사용자 제품의 강점이 얼마나 맞는지 계산 (모델 없이 순수 계산)

### 사용 데이터

| 구분 | 데이터 | 출처 |
|------|--------|------|
| 국가 리뷰 키워드 | Top10 상품의 top_keywords 합산 | `ReviewAnalysisCache` |
| 국가 카테고리 성향 | Top10 상품의 category_scores 평균 | `ReviewAnalysisCache` |
| 사용자 측 | 주요 성분, 핵심 효능 | 사용자 입력 |

### 계산 방법

**1. 키워드 오버랩 점수 (가중치 0.6)**
```python
# 국가 Top10 상품 키워드 풀 집계
country_keywords = ["보습", "진정", "흡수력", "향", "발림성"]

# 사용자 입력과 교집합
user_keywords = ["장벽 강화", "진정", "보습"]
overlap = len(set(user_keywords) & set(country_keywords))
keyword_score = overlap / len(country_keywords)
```

**2. 카테고리 성향 점수 (가중치 0.4)**
```python
# 사용자 입력 → 규칙 테이블 → 관련 카테고리 추출
KEYWORD_CATEGORY_MAP = {
    "보습": "효과_성분",    "수분": "효과_성분",
    "진정": "효과_성분",    "장벽": "효과_성분",
    "발림성": "사용감_텍스처", "흡수": "사용감_텍스처",
    "향": "향_냄새",        "냄새": "향_냄새",
    "자극": "피부_트러블_부작용", "민감": "피부_트러블_부작용",
    "지속력": "지속력_밀착력",
    "가성비": "가격_가성비",
}
# 관련 카테고리의 국가 평균 만족도 점수 (0~5) → 정규화 (÷5)
category_score = avg(category_scores[관련_카테고리]) / 5.0
```

**3. 최종 합산**
```
리뷰 유사도 = 키워드_오버랩(0.6) + 카테고리_성향(0.4)
```

**4. 정규화 (국가 간 상대 비율)**
```
국가A 정규화 값 = 국가A 값 / (국가A 값 + 국가B 값)
```

### 예외 처리

| 상황 | 처리 방법 |
|------|----------|
| ReviewAnalysisCache 없음 | 0.5 중립값 적용 |
| 카테고리 매핑 키워드 없음 | 카테고리 성향 점수 0.5 적용 |
| Top10 상품 리뷰 없음 | 0.5 중립값 적용 |
